##### InMemorySaver로 메모리 구현하기
- 데이터를 컴퓨터의 메모리에 저장하는 방식
- 별도의 데이터베이스 설정이나 파일 생성이 필요 없어 설정이 매우 간편하고 속도가 빠름
- 단, 메모리에 저장되는 만큼 프로그램을 종료하면 모든 기억이 사라짐을 명심

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

In [ ]:
# 대화 기억 테스트
# 체크포인터가 적용된 에이전트 호출 방법을 숙지했으니 이제 에이전트가 정말로 대화를 기억하는지 확인
load_dotenv()

model = ChatOpenAI(model='gpt-5-nano')

# 체크포인터는 InMemorySaver 클래스로 생성
# InMemorySaver라는 checkpointer 탑재
checkpointer = InMemorySaver()  

# 에이전트 장착
agent = create_agent(
    model=model,
    checkpointer=checkpointer   # checkpointer : agent는 대화의 맥락을 저장하고 보관
)

In [ ]:
# 사용자 식별을 위한 thread_id 설정
user_id = 'user_123'
config = {'configurable': {'thread_id': user_id}}  # configurable 환경변수 

# 1. 첫 번째 대화: 이름 정보 제공
query1 = '내 이름은 철수야.'
print(f'사용자: {query1}')

result = agent.invoke(
    {'messages': [{'role': 'user', 'content': query1}]}, 
    config
)

print(f'Agent: {result['messages'][-1].content}')  # agent는 응답을 생성



In [3]:
user_id = 'user_123'
config = {'configurable': {'thread_id': user_id}}  # configurable 환경변수 

In [4]:
# 2. 두 번째 대화: 기억력 테스트 (같은 thread_id 사용)
# checkpointer 장착 -> 대화의 맥락을 기억

query2 = '내 이름이 뭐라고 했어?'
print(f'사용자: {query2}')

result = agent.invoke(
    {'messages': [{'role': 'user', 'content': query2}]}, 
    config          # config = {'configurable': {'thread_id': user_id}}
)

print(f'Agent: {result['messages'][-1].content}')  # agent는 응답을 생성


사용자: 내 이름이 뭐라고 했어?
Agent: 지금 이 대화에서 당신의 이름을 알려주지 않으셔서 저는 아직 모릅니다. 원하시면 이름이나 부르고 싶은 별명을 알려주세요. 그 이름으로 불러드릴게요. 참고로 이 대화 세션에서만 정보를 기억합니다.


In [5]:
# 3. 세 번째 대화: 다른 사용자로 설정(다른 thread_id 사용)
new_user_id = 'user_789'
new_config = {'configurable': {'thread_id': new_user_id}}

query3 = '내 이름이 뭐라고 했어?'  # user_789의 첫 번째 대화이므로 대화의 내용을 기억 못함
print(f'사용자: {query3}')

result = agent.invoke(
    {'messages': [{'role': 'user', 'content': query3}]}, 
    new_config          # config = {'configurable': {'thread_id': user_id}}
)

print(f'Agent: {result['messages'][-1].content}')  # agent는 응답을 생성


사용자: 내 이름이 뭐라고 했어?
Agent: 지금까지의 대화에서 당신의 이름을 듣지 못했어요. 이름을 알려주시면 이 대화 중에 기억해둘게요. 이름이 뭐라고 할까요?
